# EDA

## Import Libs

In [ ]:
import os
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

## Define paths

In [ ]:
notebook_dir = Path.cwd()
base_path = notebook_dir.parent.parent / "Data" 
base_path_str = str(base_path)
print(f"Base path automatically set to: {base_path_str}")

In [ ]:
# Load .shp file
label_dir = os.path.join(base_path, "9_Training_Data")
gdf = gpd.read_file(os.path.join(label_dir,"training_data_final.shp"))

## Explore Class distributions

### Class Overview
	- 10: Vineyard (Vital)
	- 15: Vineyard (Burned)
	- 20: OliveTree
	- 30: Bare Soil
	- 40: Burned Area


In [ ]:
print("\n--- CLASS DISTRIBUTION ---")
counts = gdf['Final_Clas'].value_counts(dropna=False)
for cls_name, count in counts.items():
    print(f"Class '{cls_name}': {count} polygons")
print("--------------------------------------\n")

In [ ]:
labeled = gdf[gdf['Final_Clas'] != -1]
notlabeled = gdf[gdf['Final_Clas'] == -1]

print(f"Number of labled polygons:   {labeled.shape[0]}, {round(labeled.shape[0] / gdf.shape[0], 4) * 100} % are labeled")
print(f"Number of unlabled polygons: {notlabeled.shape[0]}, {round(notlabeled.shape[0] / gdf.shape[0], 4) * 100} % are not labeled")

In [ ]:
# 1. Class Names
class_names = {
    10: "Vineyard (Vital)",
    15: "Vineyard (Burned)",
    20: "Olive Tree",
    30: "Bare Soil",
    40: "Burned Area"
}

# Create column with nammes
labeled = labeled.copy() 
labeled['Class_Name'] = labeled['Final_Clas'].map(class_names)

labeled['Brightness'] = labeled['meanB0'] + labeled['meanB1'] + labeled['meanB2']

# Bands we want to examine 
bands = ['meanB0', 'meanB1', 'meanB2', 'meanB3', 'meanB4', 'meanB5', 'meanB6', "Brightness"]
band_titles = ['Red (B0)', 'Green (B1)', 'Blue (B2)', 'CHM (B3)', 'ExG (B4)', 'VARI (B5)', 'NGRDI (B6)', "Brightness (B7)"]

# --- TEXT-OUTPUT (Min, Mean, Max) ---
print("="*60)
print("STATISTIC OVERVIEW OF ALL CLASSES AND BANDS")
print("="*60)

for band, title in zip(bands, band_titles):
    print(f"\n--- {title} ---")
    for cls_id, cls_name in class_names.items():
        subset = labeled[labeled['Final_Clas'] == cls_id]
        
        if len(subset) > 0:
            b_min = np.nanmin(subset[band])
            b_mean = np.nanmean(subset[band])
            b_max = np.nanmax(subset[band])
            print(f"{cls_name:<18} | Min: {b_min:>7.4f} | Mean: {b_mean:>7.4f} | Max: {b_max:>7.4f}")

# --- BOXPLOT-DASHBOARD ---
print("\Generate Boxplots...")

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(18, 15))
axes = axes.flatten()

for i, (band, title) in enumerate(zip(bands, band_titles)):

    print(band)
    print(title)
    sns.boxplot(
        data=labeled,
        x='Class_Name',
        y=band,
        ax=axes[i],
        palette='Set2', 
        showfliers=True 
    )
    
    axes[i].set_title(title, fontsize=14, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Scaled Value')
    
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].grid(axis='y', linestyle='--', alpha=0.7)

for j in range(len(bands), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## Save figures for README

In [ ]:
save_dir = os.path.join(base_path.parent, "docs", "images")

# 1. CHM (Canopy Height Model) Plot & Save
plt.figure(figsize=(10, 6))
sns.boxplot(data=labeled, x='Class_Name', y='meanB3', palette='Set2', showfliers=True)
plt.title("Canopy Height Model (CHM)", fontsize=16, fontweight='bold')
plt.xlabel('')
plt.ylabel('Scaled Value')
plt.xticks(rotation=30)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(os.path.join(save_dir, "boxplot_chm.png"), dpi=300, bbox_inches='tight')
plt.close() #

# 2. ExG (Excess Green) Plot & Save
plt.figure(figsize=(10, 6))
sns.boxplot(data=labeled, x='Class_Name', y='meanB4', palette='Set2', showfliers=True)
plt.title("Excess Green Index (ExG)", fontsize=16, fontweight='bold')
plt.xlabel('')
plt.ylabel('Scaled Value')
plt.xticks(rotation=30)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(os.path.join(save_dir, "boxplot_exg.png"), dpi=300, bbox_inches='tight')
plt.close()